<a href="https://colab.research.google.com/github/LeonardooAlves/WM9B7-AIDL/blob/main/Week%203/1_Introduction_to_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Chatbot with Transformers


**MSc Applied Artificial Intelligence — AI & Deep Learning Module**

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain the **Transformer architecture** — self-attention, positional encoding, and the encoder–decoder structure.
2. Understand how **sub-word tokenisation** works and why it replaces traditional NLP pre-processing.
3. Explore **token probabilities** and understand how language models "think."
4. Compare **decoding strategies** (greedy, beam search, top-k, nucleus sampling) and visualise their effects.
5. Build a **functional chatbot** using DialoGPT, progressing from single-turn to multi-turn dialogue.
6. Engineer **system prompts and personas** to control chatbot behaviour.
7. Evaluate chatbot outputs with qualitative and quantitative methods.
8. Discuss the ethical considerations and limitations of generative language models.


| Section | Topic | Time |
|---------|-------|------|
| 1 | Transformer Architecture | 10 min |
| 2 | Environment Setup | 5 min |
| 3 | Tokenisation Deep Dive | 10 min |
| 4 | How Language Models Generate Text | 10 min |
| 5 | Decoding Strategies | 10 min |
| 6 | Building a Chatbot with DialoGPT | 15 min |
| 7 | Prompt Engineering & Personas | 5 min |
| 8 | Evaluation | 5 min |
| 9 | Limitations & Ethics | 5 min |

# 1. What Are Transformers?

## 1.1 From RNNs to Transformers

Before Transformers, sequence modelling relied on **Recurrent Neural Networks (RNNs)** and their variants (LSTMs, GRUs). These models process tokens one at a time, left to right, which creates two fundamental problems:

- **Sequential bottleneck:** Each token must wait for the previous token to be processed, making training slow and impossible to parallelise effectively across GPUs.
- **Long-range forgetting:** Even with gating mechanisms (LSTM cell states, GRU reset gates), information from early tokens fades as the sequence grows. A 500-word document effectively "forgets" the first paragraph by the time the model reaches the last sentence.

In 2017, Vaswani et al. published *"Attention Is All You Need"*, introducing the **Transformer** — an architecture that replaces recurrence entirely with **self-attention**. Self-attention allows every token in a sequence to attend to every other token *simultaneously*, solving both problems above.

## 1.2 Key Components of the Transformer

The Transformer has three foundational mechanisms:

| Component | Purpose |
|-----------|---------|
| **Self-Attention** | Lets each token compute a weighted combination of all other tokens, capturing context regardless of distance. |
| **Positional Encoding** | Injects information about token position, since self-attention has no inherent notion of order. |
| **Encoder–Decoder Structure** | The encoder reads the input; the decoder generates the output, attending to both its own prior outputs and the encoder's representations. |

## 1.3 The Three Transformer Families

Not all Transformers are built the same. The original architecture has been adapted into three families:

| Family | Architecture | Example Models | Best For |
|--------|-------------|---------------|----------|
| **Encoder-only** | Reads text bidirectionally | BERT, RoBERTa, DistilBERT | Classification, NER, sentiment analysis |
| **Decoder-only** | Generates text left-to-right | GPT-2, GPT-4, LLaMA | Text generation, chatbots, code completion |
| **Encoder-Decoder** | Reads input → generates output | T5, BART, mBART | Translation, summarisation |

For **chatbots and text generation**, we primarily use **decoder-only** models (e.g., GPT-2, DialoGPT), which generate text auto-regressively: each new token is conditioned on all previous tokens.

## 1.4 Self-Attention in Detail

Self-attention computes three vectors for each token: a **Query (Q)**, a **Key (K)**, and a **Value (V)**. The attention score between two tokens is the dot product of one token's Query with another's Key, scaled and passed through a softmax to produce weights. These weights are then used to compute a weighted sum of the Value vectors.

**Intuition:** When processing the word `"it"` in *"The cat sat on the mat because it was tired"*, the self-attention mechanism assigns high weight to `"cat"` — learning that `"it"` refers to `"cat"` — regardless of how far apart they are in the sequence.

The formula is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

where $d_k$ is the dimension of the key vectors (the scaling factor prevents dot products from growing too large, which would push softmax into regions with tiny gradients).

**Multi-head attention** runs this computation multiple times in parallel with different learned projections, allowing the model to attend to different types of relationships simultaneously — for example, syntactic dependencies in one head and semantic similarity in another.

## 1.5 Positional Encoding

Since self-attention processes all tokens simultaneously (unlike RNNs which process sequentially), the model has no inherent notion of word order. The sentence *"Dog bites man"* would produce the same attention scores as *"Man bites dog"* without positional information.

**Positional encoding** solves this by adding a unique signal to each token's embedding based on its position. The original Transformer uses sinusoidal functions at different frequencies; modern models (GPT-2, BERT) use **learned positional embeddings** — the model learns a unique vector for each position during training.

# 2. Setting Up the Environment

We will use the [Hugging Face Transformers](https://huggingface.co/docs/transformers/) library, which provides access to thousands of pre-trained models via a unified API.

In [ ]:
# Install required packages
#!pip install transformers torch matplotlib numpy transformers[torch]

In [ ]:
#!pip install --upgrade torch torchvision torchaudio

The packages:
- `transformers` — Hugging Face's library for pre-trained models and tokenisers.
- `torch` (PyTorch) — the deep learning backend that runs model computations.
- `matplotlib` — for visualising token probabilities and attention patterns.
- `numpy` — numerical operations for processing model outputs.

In [ ]:
# Import core libraries
import torch
import torch.nn.functional as F
import numpy as np
from huggingface_hub import login
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import warnings
warnings.filterwarnings('ignore')

# Use GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


We check for GPU availability. Transformer models are compute-intensive; a GPU accelerates inference significantly. In Google Colab, you can enable a free GPU via `Runtime → Change runtime type → T4 GPU`.

In [ ]:
login(token="YOUR_TOKEN_HERE")  # paste your token from huggingface.co/settings/tokens

HfHubHTTPError: Invalid user token.

The key imports:

- `AutoTokenizer` — automatically loads the correct tokeniser for any model name.
- `AutoModelForCausalLM` — loads a **causal language model** (decoder-only Transformer) for text generation.
- `torch.nn.functional` (as `F`) — gives us `softmax` for converting raw model outputs (logits) into probabilities.
- `pipeline` — a high-level wrapper that combines tokenisation, inference, and post-processing into a single call.

# 3. Tokenisation Deep Dive

Before a Transformer can process text, it must be converted into numerical **token IDs**. Modern models use **sub-word tokenisation** (typically Byte-Pair Encoding or WordPiece), which is fundamentally different from the word-level tokenisation we saw in traditional NLP pre-processing.

## Why Sub-Word Tokenisation?

Traditional word tokenisation has two problems:
- **Vocabulary explosion:** Every unique word needs its own entry. Rare words, misspellings, and compound words create an enormous vocabulary.
- **Out-of-vocabulary (OOV) problem:** Words not in the vocabulary are mapped to a generic `[UNK]` token, losing all meaning.

Sub-word tokenisation solves both by splitting rare words into common sub-units while keeping frequent words intact. For example, `"unhappiness"` might become `["un", "happiness"]`.

**Key insight:** This means we do **not** need to remove stop words, stem, or lemmatise — the model learns to handle all of this internally.

In [ ]:
# Load the GPT-2 tokeniser
tokenizer = AutoTokenizer.from_pretrained("gpt2")

print(f"Vocabulary size: {tokenizer.vocab_size:,} tokens")
print(f"Model max length: {tokenizer.model_max_length:,} tokens")

GPT-2's vocabulary contains 50,257 sub-word tokens. This is much smaller than a full word-level vocabulary (which could be 500,000+ for English) but covers virtually all text through sub-word decomposition.

## Step 1: Tokenising a Sentence

Let's see how GPT-2 breaks text into tokens.

In [ ]:
# Tokenise a simple sentence
text = "Transformers have revolutionised natural language processing."
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print(f"Original text: {text}")
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")
print(f"Number of tokens: {len(tokens)}")

Notice that common words like `"have"` and `"natural"` are kept as single tokens, while less common words may be split. The `Ġ` prefix (visible in GPT-2 tokens) indicates a space before that token — this is how GPT-2 preserves whitespace information.

Each token maps to a unique integer ID in the vocabulary. These IDs are what the model actually processes — it never sees the original text.

## Step 2: Exploring Common vs. Uncommon Words

Let's compare how the tokeniser handles different types of words.

In [ ]:
# Compare tokenisation of common vs. uncommon words
test_words = [
    "the",                        # very common
    "computer",                   # common
    "artificial",                 # moderately common
    "supercalifragilistic",       # rare / made-up
    "ChatGPT",                    # modern term (not in GPT-2 training data)
    "COVID-19",                   # post-training term
    "🤖",                        # emoji
]

print(f"{'Word':<30} {'Tokens':<40} {'# Tokens'}")
print("=" * 80)
for word in test_words:
    tokens = tokenizer.tokenize(word)
    print(f"{word:<30} {str(tokens):<40} {len(tokens)}")

**What to observe:**

- **Common words** (`"the"`, `"computer"`) map to a single token.
- **Rare or compound words** get split into sub-word pieces.
- **Post-training terms** like `"ChatGPT"` and `"COVID-19"` get decomposed because they didn't exist when GPT-2 was trained (2019). The model still processes them, but as separate pieces rather than a single concept.
- **Emojis** are encoded as byte-level tokens — GPT-2's Byte-Pair Encoding can handle any UTF-8 character.

This demonstrates a key property: sub-word tokenisation **never fails** on unknown input. It always produces some representation, even for words the model has never seen.

## Step 3: Encoding and Decoding Round-Trip

In [ ]:
# Demonstrate encoding and decoding round-trip
text = "Hello, how are you today?"
encoded = tokenizer.encode(text, return_tensors="pt")
decoded = tokenizer.decode(encoded[0])

print(f"Original:      '{text}'")
print(f"Encoded shape: {encoded.shape}  (1 sequence x {encoded.shape[1]} tokens)")
print(f"Encoded IDs:   {encoded[0].tolist()}")
print(f"Decoded back:  '{decoded}'")

The encoding–decoding round trip is **lossless**: `encode()` converts text → token IDs, and `decode()` converts IDs → text, recovering the exact original string. The tensor shape `[1, N]` means 1 sequence of N tokens — this is the format the model expects as input.

## Step 4: Words vs. Tokens — the Ratio Matters

A common misconception is that 1 word ≈ 1 token. In practice, a word averages about 1.3 tokens in English. Understanding this ratio matters because models have a **maximum context length** measured in tokens, not words.

In [ ]:
# Compare word count vs token count across different text styles
examples = {
    "Simple English": "The cat sat on the mat and looked at the bird outside.",
    "Technical jargon": "The convolutional neural network's backpropagation algorithm optimises hyperparameters.",
    "Informal/slang": "lol thats sooo cool!! cant believe u did that hahaha",
    "Code-like": "def calculate_loss(predictions, targets): return F.cross_entropy(predictions, targets)",
}

print(f"{'Type':<25} {'Words':<8} {'Tokens':<8} {'Ratio'}")
print("=" * 55)
for name, text in examples.items():
    words = len(text.split())
    tokens = len(tokenizer.tokenize(text))
    print(f"{name:<25} {words:<8} {tokens:<8} {tokens/words:.2f}")

**Key observations:**

- Simple, common English is close to 1:1 (words to tokens).
- Technical vocabulary inflates the token count because specialised terms are split into sub-words.
- Code-like text tends to have a high token ratio due to special characters and naming conventions.

**Why this matters:** GPT-2 has a context window of **1,024 tokens**. At ~1.3 tokens per word, that's roughly 780 words — about one page of text. As a conversation grows, older turns get pushed out of this window and the model literally *forgets* them. Modern models like GPT-4 (128k tokens) and Claude (200k tokens) have much larger windows, dramatically reducing this problem.

## Step 5: Tokenising an Entire Paragraph

Let's see the complete tokenisation pipeline on a longer piece of text.

In [ ]:
# Full tokenisation pipeline on a paragraph
paragraph = """Machine learning is a subset of artificial intelligence that enables
systems to learn and improve from experience. Instead of being explicitly
programmed, these systems use algorithms to identify patterns in data."""

# Step-by-step
tokens = tokenizer.tokenize(paragraph)
ids = tokenizer.encode(paragraph)
reconstructed = tokenizer.decode(ids)

print(f"Original ({len(paragraph.split())} words):")
print(paragraph)
print(f"\nTokenised ({len(tokens)} tokens):")
print(tokens)
print(f"\nToken IDs:")
print(ids)
print(f"\nReconstructed:")
print(reconstructed)

Note how the paragraph was broken into individual sub-word tokens, each mapped to an integer ID, and then perfectly reconstructed. This pipeline — text → tokens → IDs → model → IDs → tokens → text — is the fundamental loop behind every Transformer application.

# 4. How Language Models Generate Text

## 4.1 Loading GPT-2

GPT-2 is a **decoder-only Transformer** pre-trained by OpenAI on WebText — a dataset of ~8 million web pages. It generates text by predicting the **next token** given all previous tokens, a process called **auto-regressive generation**.

In [ ]:
# Load GPT-2 (small, 124M parameters)
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {model_name}")
print(f"Parameters: {total_params:,}")

GPT-2 small has approximately 124 million parameters. We did **not** train this model — we are using **transfer learning**, leveraging a model that already learned the statistical patterns of English from billions of words of text.

In [ ]:
# Inspect model architecture
print(f"Layers: {model.config.n_layer}")
print(f"Attention heads per layer: {model.config.n_head}")
print(f"Hidden dimension: {model.config.n_embd}")
print(f"Context window: {model.config.n_positions} tokens")
print(f"Vocabulary size: {model.config.vocab_size:,}")

These 124 million parameters are distributed across 12 Transformer layers, each with 12 attention heads and a hidden dimension of 768. The context window of 1,024 tokens defines the maximum sequence the model can attend to at once.

## 4.2 Peeking Inside: What Does the Model Predict?

Before generating text, let's understand what the model actually computes. Given a sequence of tokens, the model outputs a **probability distribution** over its entire vocabulary (50,257 tokens) for what comes next.

In [ ]:
# See what the model predicts as the next token
prompt = "The capital of France is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

# Get model outputs (logits = raw scores before softmax)
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits  # shape: [1, seq_len, vocab_size]

print(f"Input shape: {input_ids.shape}")
print(f"Output logits shape: {logits.shape}")
print(f"\nThe model produces a distribution over {logits.shape[-1]:,} tokens at EACH position.")

The logits tensor has shape `[1, 6, 50257]` — for each of the 6 input tokens, the model predicts a distribution over all 50,257 vocabulary tokens. We care about the **last position** — that's the prediction for the **next** token after our prompt.

In [ ]:
# Get the logits for the LAST position (predicting the next token)
next_token_logits = logits[0, -1, :]  # shape: [vocab_size]

# Convert raw logits to probabilities using softmax
probs = F.softmax(next_token_logits, dim=-1)

print(f"Logits range: [{next_token_logits.min().item():.2f}, {next_token_logits.max().item():.2f}]")
print(f"Probabilities sum to: {probs.sum().item():.4f} (should be 1.0)")

The raw logits are arbitrary real numbers. The `softmax` function converts them into a valid probability distribution: all values between 0 and 1, summing to 1.0. This is the distribution we sample from when generating text.

In [ ]:
# Get top 10 most probable next tokens
top_k_probs, top_k_indices = torch.topk(probs, 10)

print(f"Prompt: '{prompt}'")
print(f"\nTop 10 predicted next tokens:")
print(f"{'Rank':<6} {'Token':<20} {'Probability':<15}")
print("=" * 45)
for i, (prob, idx) in enumerate(zip(top_k_probs, top_k_indices)):
    token = tokenizer.decode(idx)
    print(f"{i+1:<6} '{token}'{'':>{18-len(token)}} {prob.item():.4f} ({prob.item()*100:.1f}%)")

The model assigns the highest probability to the correct answer. But notice that other tokens also have non-zero probability. This probability distribution is the **foundation of all text generation** — the decoding strategy determines which token is actually selected.

## 4.3 Visualising the Probability Distribution

In [ ]:
# Bar chart: top 15 tokens
top_n = 15
top_probs, top_indices = torch.topk(probs, top_n)
top_tokens = [tokenizer.decode(idx).strip() for idx in top_indices]
top_probs_np = top_probs.cpu().numpy()

plt.figure(figsize=(10, 5))
plt.barh(range(top_n), top_probs_np, color='steelblue', edgecolor='black')
plt.yticks(range(top_n), top_tokens)
plt.gca().invert_yaxis()
plt.xlabel("Probability")
plt.title(f"Top {top_n} Next-Token Probabilities — Prompt: '{prompt}'")
plt.tight_layout()
plt.show()

The bar chart shows the model's top candidates. The distribution is very peaked — one or two tokens dominate, with a steep drop-off. This is typical when the context strongly constrains the next word.

In [ ]:
# Full distribution: the long tail
all_probs_sorted = torch.sort(probs, descending=True).values.cpu().numpy()

plt.figure(figsize=(10, 4))
plt.plot(range(len(all_probs_sorted)), all_probs_sorted, color='steelblue', linewidth=0.5)
plt.yscale('log')
plt.xlabel("Token rank (sorted by probability)")
plt.ylabel("Probability (log scale)")
plt.title("Full Vocabulary Distribution (50,257 tokens)")
plt.axhline(y=1/50257, color='red', linestyle='--', alpha=0.5, label='Uniform probability')
plt.legend()
plt.tight_layout()
plt.show()

The full distribution across all 50,257 tokens shows a massive **long tail** on a log scale. The vast majority of tokens have near-zero probability. The red dashed line shows what a uniform (completely random) distribution would look like.

In [ ]:
# Summary statistics
print(f"Highest probability: {probs.max().item():.4f}")
print(f"Tokens with >1% probability: {(probs > 0.01).sum().item()}")
print(f"Tokens with >0.1% probability: {(probs > 0.001).sum().item()}")
entropy = -(probs * torch.log2(probs + 1e-10)).sum().item()
print(f"Entropy: {entropy:.2f} bits")
print(f"(Uniform entropy would be: {np.log2(50257):.2f} bits)")

**Entropy** measures the model's uncertainty. Low entropy = confident (few likely tokens). High entropy = uncertain (many plausible tokens). A uniform distribution over 50,257 tokens would have ~15.6 bits of entropy; our factual prompt produces much lower entropy because the model is confident about the answer.

## 4.4 How Confidence Changes with Context

More context generally means more confident predictions. Let's verify this.

In [ ]:
# Compare model confidence across different prompts
prompts = [
    "The",                                    # very ambiguous
    "The capital",                            # somewhat constrained
    "The capital of France",                  # more specific
    "The capital of France is",               # very constrained
]

print(f"{'Prompt':<40} {'Top Token':<12} {'Top Prob':<10} {'Entropy'}")
print("=" * 75)
for prompt in prompts:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(input_ids).logits[0, -1, :]
    p = F.softmax(logits, dim=-1)
    top_prob, top_idx = p.max(dim=-1)
    ent = -(p * torch.log2(p + 1e-10)).sum().item()
    top_token = tokenizer.decode(top_idx).strip()
    print(f"'{prompt}'{'':>{38-len(prompt)}} {top_token:<12} {top_prob.item():.4f}     {ent:.2f} bits")

As the prompt becomes more specific, the top token probability increases and entropy decreases — the model becomes more certain. This pattern holds generally: ambiguous prompts have many plausible continuations (high entropy), while constrained prompts have few (low entropy).

In [ ]:
# Now try open-ended vs factual prompts
prompts2 = [
    "I went to the store and bought some",    # open-ended
    "The chemical formula for water is",       # highly constrained
    "Once upon a time there was a",            # creative, open
    "The first president of the United States was", # factual
]

print(f"{'Prompt':<55} {'Top Token':<12} {'Top Prob':<10} {'Entropy'}")
print("=" * 90)
for prompt in prompts2:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(input_ids).logits[0, -1, :]
    p = F.softmax(logits, dim=-1)
    top_prob, top_idx = p.max(dim=-1)
    ent = -(p * torch.log2(p + 1e-10)).sum().item()
    top_token = tokenizer.decode(top_idx).strip()
    print(f"'{prompt}'{'':>{53-len(prompt)}} {top_token:<12} {top_prob.item():.4f}     {ent:.2f} bits")

**Open-ended prompts** (shopping, storytelling) produce high entropy — many reasonable continuations exist. **Factual prompts** (chemical formulae, historical facts) produce low entropy — the model is confident.

This directly impacts chatbot design: open-ended conversation has high entropy, which is why the choice of sampling strategy matters more for chatbots than for factual question-answering.

# 5. Decoding Strategies

Given the probability distribution, we need a strategy to select which token to actually generate. This choice dramatically affects the quality, diversity, and coherence of the output.

## 5.1 Greedy Decoding

The simplest strategy: always pick the most probable next token. Deterministic — same prompt always produces the same output.

In [ ]:
# Greedy generation
prompt = "Artificial intelligence will"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

output = model.generate(
    input_ids,
    max_new_tokens=60,
    do_sample=False,        # greedy: always pick highest-probability token
    pad_token_id=tokenizer.eos_token_id
)

print("GREEDY DECODING:")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Greedy decoding often produces grammatically correct but *repetitive* text. The highest-probability token at each step leads to safe, common continuations, creating a feedback loop that causes loops and generic phrasing.

In [ ]:
# Run greedy twice to confirm it's deterministic
for i in range(3):
    output = model.generate(input_ids, max_new_tokens=30, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
    print(f"Run {i+1}: {tokenizer.decode(output[0], skip_special_tokens=True)}")

All three runs produce identical output — greedy decoding is fully deterministic. This is useful for reproducibility but problematic for chatbots, where we want varied, natural-sounding responses.

## 5.2 Temperature Scaling

**Temperature** scales the logits before softmax, controlling how "peaked" or "flat" the probability distribution is:

- **Low temperature (< 1.0):** Sharpens the distribution → more deterministic and conservative.
- **Temperature = 1.0:** Raw distribution as-is (default).
- **High temperature (> 1.0):** Flattens the distribution → more random and creative.

In [ ]:
# Visualise the effect of temperature
prompt = "I went to the store and bought some"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    logits = model(input_ids).logits[0, -1, :]

temperatures = [0.3, 0.7, 1.0, 1.5, 2.0]
fig, axes = plt.subplots(1, len(temperatures), figsize=(18, 4), sharey=True)

for ax, temp in zip(axes, temperatures):
    scaled_probs = F.softmax(logits / temp, dim=-1)
    top_probs, top_indices = torch.topk(scaled_probs, 10)
    tokens = [tokenizer.decode(idx).strip() for idx in top_indices]

    ax.barh(range(10), top_probs.cpu().numpy(), color='steelblue', edgecolor='black')
    ax.set_yticks(range(10))
    ax.set_yticklabels(tokens, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(f"Temp = {temp}")
    ax.set_xlabel("Probability")
    entropy = -(scaled_probs * torch.log2(scaled_probs + 1e-10)).sum().item()
    ax.text(0.95, 0.95, f"H={entropy:.1f}", transform=ax.transAxes, ha='right', va='top', fontsize=9)

plt.suptitle(f"Effect of Temperature on Next-Token Distribution", fontsize=13)
plt.tight_layout()
plt.show()

**Reading the visualisation:**

- At **temperature 0.3**, one token dominates — the model is nearly deterministic.
- At **temperature 1.0** (default), the distribution reflects the model's trained probabilities.
- At **temperature 2.0**, the distribution is nearly flat — essentially choosing randomly.
- **H** (entropy) increases with temperature, confirming higher temperature = more randomness.

**Practical guideline:** Factual chatbots → low temperature (0.3–0.7). Creative applications → moderate temperature (0.8–1.2). Above 1.5 tends to produce incoherent text.

In [ ]:
# Generate text at different temperatures to see the effect
prompt = "The best way to learn programming is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

for temp in [0.3, 0.7, 1.0, 1.5]:
    output = model.generate(input_ids, max_new_tokens=40, do_sample=True,
                            temperature=temp, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Temp {temp}: {text}")
    print()

Observe how low temperature produces safe, predictable continuations while high temperature introduces more surprising (but potentially less coherent) word choices.

## 5.3 Top-k Sampling

Instead of adjusting the shape of the distribution, **top-k sampling** truncates it: only the k most probable tokens are kept, and the rest are set to zero probability.

In [ ]:
# Visualise top-k truncation
prompt = "I went to the store and bought some"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    logits = model(input_ids).logits[0, -1, :]
probs = F.softmax(logits, dim=-1)
sorted_probs, sorted_indices = torch.sort(probs, descending=True)

k = 50
plt.figure(figsize=(10, 4))
plt.bar(range(80), sorted_probs[:80].cpu().numpy(),
        color=['steelblue' if i < k else 'lightgray' for i in range(80)], edgecolor='none')
plt.axvline(x=k-0.5, color='red', linestyle='--', label=f'k={k} cutoff')
plt.xlabel("Token rank")
plt.ylabel("Probability")
plt.title(f"Top-k Sampling (k={k}): tokens beyond the red line are excluded")
plt.legend()
plt.tight_layout()
plt.show()

Blue bars are tokens that can be selected; grey bars are excluded. Top-k always keeps exactly k tokens, regardless of the model's confidence.

## 5.4 Nucleus (Top-p) Sampling

**Nucleus sampling** takes a different approach: keep the smallest set of tokens whose cumulative probability exceeds a threshold p. This adapts dynamically — fewer candidates when the model is confident, more when uncertain.

In [ ]:
# Visualise nucleus (top-p) truncation
cumulative = sorted_probs.cumsum(dim=0).cpu().numpy()
p = 0.9
cutoff_idx = np.searchsorted(cumulative, p)

colors = ['steelblue' if i <= cutoff_idx else 'lightgray' for i in range(80)]
plt.figure(figsize=(10, 4))
plt.bar(range(80), sorted_probs[:80].cpu().numpy(), color=colors, edgecolor='none')
plt.axvline(x=cutoff_idx + 0.5, color='red', linestyle='--',
            label=f'p={p} cutoff ({cutoff_idx+1} tokens)')
plt.xlabel("Token rank")
plt.ylabel("Probability")
plt.title(f"Nucleus Sampling (p={p}): {cutoff_idx+1} tokens cover {p*100:.0f}% of probability mass")
plt.legend()
plt.tight_layout()
plt.show()

**Key difference from top-k:** Nucleus sampling adapts. When the model is confident (90% probability on one token), only a few tokens pass the threshold. When uncertain, many tokens are included. This makes nucleus sampling generally more robust across different contexts.

**Most production chatbots use nucleus sampling** (p=0.9 or 0.95) combined with moderate temperature (0.7–0.9).

## 5.5 Comparing All Strategies Side by Side

In [ ]:
# Generate text with each strategy
prompt = "The future of education is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

strategies = {
    "Greedy": {"do_sample": False},
    "Temp=0.5 (conservative)": {"do_sample": True, "temperature": 0.5},
    "Temp=1.2 (creative)": {"do_sample": True, "temperature": 1.2},
    "Top-k (k=50)": {"do_sample": True, "top_k": 50},
    "Nucleus (p=0.9)": {"do_sample": True, "top_p": 0.9},
    "Production combo": {"do_sample": True, "temperature": 0.8, "top_p": 0.95, "top_k": 50},
}

for name, params in strategies.items():
    output = model.generate(input_ids, max_new_tokens=50,
                            pad_token_id=tokenizer.eos_token_id, **params)
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"\n{'='*70}")
    print(f"Strategy: {name}")
    print(f"{'='*70}")
    print(text)

**What to observe:**

- **Greedy** — deterministic, often repetitive.
- **Low temperature** — close to greedy with slight variation.
- **High temperature** — creative but potentially incoherent.
- **Top-k** — limits the token pool, preventing bizarre outputs.
- **Nucleus** — adapts dynamically, generally the best single strategy.
- **Production combo** — what most deployed chatbots use.

There is no single "best" strategy — it depends on the use case.

## 5.6 Exercise: Experiment with Decoding

Modify the parameters below to answer:
1. What happens with temperature = 0.1? Temperature = 3.0?
2. Compare top_k=5 vs top_k=500 — what changes?
3. Find a combination that produces good results for the prompt *"Tips for learning programming:"*

In [ ]:
# ============================================================
# EXERCISE: Modify the parameters and observe the changes
# ============================================================
prompt = "Tips for learning programming:"

input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

output = model.generate(
    input_ids,
    max_new_tokens=80,
    do_sample=True,         # set to False for greedy
    temperature=0.8,        # try: 0.1, 0.5, 0.8, 1.2, 3.0
    top_p=0.9,              # try: 0.5, 0.9, 0.95, 1.0
    top_k=50,               # try: 5, 50, 500
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

# 6. Building a Conversational Chatbot

Now we move from raw text generation to **dialogue**. A chatbot must handle **multi-turn conversation** — remembering what was said and responding coherently.

## 6.1 DialoGPT: A Dialogue-Optimised Model

**DialoGPT** (Microsoft) is GPT-2 fine-tuned on 147 million Reddit comment chains. It understands conversational patterns that plain GPT-2 does not.

| Aspect | GPT-2 | DialoGPT |
|--------|-------|----------|
| Training data | WebText (general web) | Reddit comment threads |
| Understands turn-taking | No | Yes |
| Conversational responses | Sometimes | Consistently |
| Available sizes | 124M, 355M, 774M, 1.5B | 124M, 345M, 762M |

In [ ]:
# Load DialoGPT-medium
chatbot_name = "microsoft/DialoGPT-medium"
chat_tokenizer = AutoTokenizer.from_pretrained(chatbot_name)
chat_model = AutoModelForCausalLM.from_pretrained(chatbot_name).to(device)

chat_tokenizer.pad_token = chat_tokenizer.eos_token

total_params = sum(p.numel() for p in chat_model.parameters())
print(f"Model: {chatbot_name}")
print(f"Parameters: {total_params:,}")

DialoGPT-medium has approximately 345 million parameters. The key difference from GPT-2 is the **training data**: fine-tuning on conversational data teaches the model turn-taking, question answering, and conversational flow.

## 6.2 Single-Turn Chat

Let's start with the simplest case: one question → one answer, no memory.

In [ ]:
def chat_single_turn(user_input, temperature=0.75, top_p=0.9):
    """Generate a single response with no conversation history."""
    input_ids = chat_tokenizer.encode(
        user_input + chat_tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    output = chat_model.generate(
        input_ids,
        max_new_tokens=100,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=chat_tokenizer.eos_token_id
    )

    response = chat_tokenizer.decode(
        output[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )
    return response

The function works in three steps:

1. **Encode** the user input with an `eos_token` appended — signalling the user's turn is complete.
2. **Generate** continuation tokens using the model.
3. **Decode** only the *new* tokens (everything after the input) to extract just the response.

In [ ]:
# Test single-turn responses
test_inputs = [
    "What is machine learning?",
    "Tell me a joke",
    "What should I study to get into AI?",
    "What's the best programming language?",
    "How do I stay motivated when learning something difficult?",
]

for msg in test_inputs:
    response = chat_single_turn(msg)
    print(f"User: {msg}")
    print(f"Bot:  {response}")
    print()

Each response is independent — if you ask *"What is AI?"* and then *"Tell me more"*, the second call has no idea what "more" refers to. This is why we need multi-turn conversation with history.

## 6.3 GPT-2 vs DialoGPT: Direct Comparison

Let's see how a general-purpose model and a dialogue-tuned model handle the same conversational inputs.

In [ ]:
# Compare GPT-2 vs DialoGPT on the same prompts
comparison_prompts = [
    "What's your favourite book?",
    "Can you help me with my homework?",
    "I'm feeling stressed about my exams.",
]

for prompt in comparison_prompts:
    # GPT-2 response
    gpt2_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    gpt2_out = model.generate(gpt2_ids, max_new_tokens=50, do_sample=True,
                               temperature=0.8, top_p=0.9,
                               pad_token_id=tokenizer.eos_token_id)
    gpt2_response = tokenizer.decode(
        gpt2_out[0][gpt2_ids.shape[-1]:], skip_special_tokens=True
    )

    # DialoGPT response
    dialog_ids = chat_tokenizer.encode(
        prompt + chat_tokenizer.eos_token, return_tensors="pt"
    ).to(device)
    dialog_out = chat_model.generate(dialog_ids, max_new_tokens=50, do_sample=True,
                                      temperature=0.8, top_p=0.9,
                                      pad_token_id=chat_tokenizer.eos_token_id)
    dialog_response = chat_tokenizer.decode(
        dialog_out[0][dialog_ids.shape[-1]:], skip_special_tokens=True
    )

    print(f"Prompt: \"{prompt}\"")
    print(f"  GPT-2:     {gpt2_response[:150]}")
    print(f"  DialoGPT:  {dialog_response[:150]}")
    print()

**What to observe:**

- **GPT-2** tends to *continue writing* as if the prompt is the beginning of an article. It doesn't understand that a question expects a conversational answer.
- **DialoGPT** produces responses that feel like a reply — shorter, more direct, structured as an answer.

This illustrates the power of **fine-tuning**: both share the same architecture, but training on conversation data fundamentally changed the model's behaviour.

## 6.4 Multi-Turn Chat: How Conversation History Works

To build a real chatbot, we concatenate all previous turns and feed the entire history as input. The model's self-attention can then attend to everything said so far.

The mechanism works like this:

```
Turn 1: [User₁][EOS] → generates → [Bot₁][EOS]
Turn 2: [User₁][EOS][Bot₁][EOS][User₂][EOS] → generates → [Bot₂][EOS]
Turn 3: [User₁][EOS][Bot₁][EOS][User₂][EOS][Bot₂][EOS][User₃][EOS] → generates → [Bot₃][EOS]
```

Each turn, the input grows longer. When it exceeds the model's context window (1,024 tokens), we must truncate the oldest tokens — the model literally forgets early conversation.

In [ ]:
def chat_multi_turn():
    """Interactive multi-turn chatbot with conversation history."""
    print("=" * 60)
    print("  DialoGPT Chatbot")
    print("  Type 'quit' to exit | 'reset' to clear history")
    print("=" * 60)
    print()

    chat_history_ids = None
    turn_count = 0

    while turn_count < 15:
        user_input = input("You: ")

        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("Bot: Goodbye!")
            break

        if user_input.lower() == 'reset':
            chat_history_ids = None
            turn_count = 0
            print("[Conversation history cleared]\n")
            continue

        # Encode user input
        new_input_ids = chat_tokenizer.encode(
            user_input + chat_tokenizer.eos_token,
            return_tensors="pt"
        ).to(device)

        # Append to history
        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        else:
            bot_input_ids = new_input_ids

        # Truncate if exceeding context window
        max_length = 1000
        if bot_input_ids.shape[-1] > max_length:
            bot_input_ids = bot_input_ids[:, -max_length:]

        # Generate
        chat_history_ids = chat_model.generate(
            bot_input_ids,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.75,
            top_p=0.9,
            pad_token_id=chat_tokenizer.eos_token_id
        )

        response = chat_tokenizer.decode(
            chat_history_ids[0][bot_input_ids.shape[-1]:],
            skip_special_tokens=True
        )

        turn_count += 1
        token_count = chat_history_ids.shape[-1]
        print(f"Bot: {response}  [turn {turn_count} | {token_count} tokens used]")
        print()

# Uncomment to run interactively:
# chat_multi_turn()

**Key features of this implementation:**

- **History concatenation** — each turn appends to the full conversation tensor.
- **Truncation** (`max_length=1000`) — when context exceeds 1,000 tokens, the oldest turns are dropped. This prevents errors but causes the bot to forget early conversation.
- **Token counter** — displayed after each response so you can see the context filling up.
- **Reset command** — clears history to start fresh.

## 6.5 Scripted Multi-Turn Demo

Since `input()` can behave inconsistently in Colab, here is a scripted conversation so everyone can see multi-turn memory in action.

In [ ]:
# Scripted multi-turn conversation
scripted_messages = [
    "Hi! What's your name?",
    "Do you like music?",
    "What's your favourite genre?",
    "Can you recommend a song?",
    "Thanks! What about movies?",
    "What genre of movies do you prefer?",
]

chat_history_ids = None
print("=" * 60)
print("  Scripted Multi-Turn Conversation Demo")
print("=" * 60)

for user_msg in scripted_messages:
    new_input_ids = chat_tokenizer.encode(
        user_msg + chat_tokenizer.eos_token, return_tensors="pt"
    ).to(device)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    chat_history_ids = chat_model.generate(
        bot_input_ids, max_new_tokens=80, do_sample=True,
        temperature=0.75, top_p=0.9,
        pad_token_id=chat_tokenizer.eos_token_id
    )

    response = chat_tokenizer.decode(
        chat_history_ids[0][bot_input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    print(f"\nYou: {user_msg}")
    print(f"Bot: {response}")
    print(f"  [Context: {chat_history_ids.shape[-1]} tokens]")

**Observe the context length growing** with each turn. Also notice whether the bot shows awareness of earlier conversation — does it refer back to music when asked about movies? This is multi-turn memory at work (or failing). The context grows by roughly 20–40 tokens per turn, so a 1,024-token window supports about 15–25 turns before truncation begins.

## 6.6 Exercise: Test the Chatbot's Memory

Run the cell below with a scripted conversation that tests whether the bot remembers earlier context. Try:
1. Introduce yourself by name, then later ask *"What's my name?"*
2. State a preference (e.g., *"I love pizza"*), then later ask about it.
3. Ask a question, then say *"Tell me more about that."*

In [ ]:
# ============================================================
# EXERCISE: Modify the messages below to test the bot's memory
# ============================================================
my_conversation = [
    "My name is Alex and I love football.",
    "What sport do I like?",
    "What's my name?",
    "Can you recommend a football team for me?",
]

chat_history_ids = None
for user_msg in my_conversation:
    new_input_ids = chat_tokenizer.encode(
        user_msg + chat_tokenizer.eos_token, return_tensors="pt"
    ).to(device)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    chat_history_ids = chat_model.generate(
        bot_input_ids, max_new_tokens=80, do_sample=True,
        temperature=0.75, top_p=0.9,
        pad_token_id=chat_tokenizer.eos_token_id
    )

    response = chat_tokenizer.decode(
        chat_history_ids[0][bot_input_ids.shape[-1]:],
        skip_special_tokens=True
    )
    print(f"You: {user_msg}")
    print(f"Bot: {response}\n")

**Discussion:** Did the bot remember your name? Your preferences? DialoGPT has a limited ability to use context — it sometimes recalls, sometimes doesn't. Modern chatbots (ChatGPT, Claude) are specifically trained to track and use conversation context reliably, which is one reason they feel much more natural in extended conversations.

# 7. Prompt Engineering and Personas

## 7.1 Steering Behaviour with Context

Although DialoGPT doesn't support formal system prompts (unlike modern models), we can **steer** its behaviour by prepending context. This is an early form of **prompt engineering**.

In [ ]:
# Demonstrate prompt-based steering
personas = {
    "Pirate": "You are a friendly pirate who speaks in pirate slang. User: What's the weather like today?",
    "Scientist": "You are a careful scientist who always references evidence. User: Is coffee good for you?",
    "Poet": "You are a poet who speaks in beautiful metaphors. User: Tell me about the ocean.",
    "Default (no persona)": "What's the weather like today?",
}

for name, prompt in personas.items():
    input_ids = chat_tokenizer.encode(
        prompt + chat_tokenizer.eos_token, return_tensors="pt"
    ).to(device)

    output = chat_model.generate(
        input_ids, max_new_tokens=80, do_sample=True,
        temperature=0.85, top_p=0.9,
        pad_token_id=chat_tokenizer.eos_token_id
    )

    response = chat_tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True)
    print(f"[{name}]")
    print(f"  Response: {response}\n")

The persona prefix may or may not significantly steer the output. DialoGPT predates the instruction-following paradigm, so it's less responsive to system prompts than modern chatbots. This highlights an important evolution:

| Generation | Models | Prompt Following |
|-----------|--------|-----------------|
| **2019** | GPT-2, DialoGPT | Weak — must "trick" the model with context |
| **2022** | InstructGPT, ChatGPT | Strong — trained to follow instructions via RLHF |
| **2023–25** | GPT-4, Claude, LLaMA 3 | Very strong — complex personas, tool use, structured output |

Modern chatbots use **Reinforcement Learning from Human Feedback (RLHF)** and **Constitutional AI** to make instruction-following reliable.

## 7.2 The Pipeline API for Quick Prototyping

Hugging Face's `pipeline` wraps everything into a single call — useful for rapid experiments.

In [ ]:
# High-level pipeline
generator = pipeline(
    "text-generation",
    model="gpt2",
    device=0 if torch.cuda.is_available() else -1
)

results = generator(
    "The key advantage of Transformers over RNNs is",
    max_new_tokens=60,
    num_return_sequences=3,
    do_sample=True,
    temperature=0.8,
    top_p=0.9
)

for i, result in enumerate(results):
    print(f"\n--- Completion {i+1} ---")
    print(result['generated_text'])

`num_return_sequences=3` generates three independent completions from the same prompt. This enables **best-of-N sampling** — generate N candidates, then pick the best one. Production systems routinely generate 4–16 candidates and use a separate ranking model to select the response shown to the user.

# 8. Evaluating Chatbot Quality

## 8.1 Qualitative Criteria

Evaluating chatbots is notoriously difficult because there is no single "correct" answer:

| Criterion | Question to Ask | Example of Failure |
|-----------|---------------|-------------------|
| **Fluency** | Is it grammatically correct? | *"The weather is be good tomorrow"* |
| **Relevance** | Does it address the question? | Q: *"What time is it?"* A: *"I like pizza"* |
| **Coherence** | Is it internally consistent? | *"I love cats. I hate all animals."* |
| **Engagement** | Is it interesting? | *"I don't know."* to every question |
| **Safety** | Is it appropriate? | Toxic, biased, or harmful content |

In [ ]:
# Test response consistency: same prompt, multiple runs
prompt = "What is the meaning of life?"
print(f"Prompt: '{prompt}'\n")

responses = []
for i in range(5):
    response = chat_single_turn(prompt)
    responses.append(response)
    print(f"Response {i+1}: {response}")

With sampling enabled, the same prompt produces different responses. Some may be insightful, others generic. Production chatbots generate multiple candidates and use a ranking model to select the best one.

In [ ]:
# Check diversity of the 5 responses
unique = len(set(responses))
print(f"Unique responses: {unique}/5")
if unique <= 2:
    print("Low diversity — consider increasing temperature or top_p")
else:
    print("Good diversity across runs")

## 8.2 Quantitative Metrics: Distinct-N

**Distinct-N** measures the proportion of unique n-grams in generated text. It detects repetitive bots — a bot that always says *"That's interesting!"* scores very low.

In [ ]:
# Calculate Distinct-1 and Distinct-2
def distinct_n(texts, n):
    """Proportion of unique n-grams across all texts."""
    all_ngrams = []
    for text in texts:
        words = text.lower().split()
        ngrams = [tuple(words[i:i+n]) for i in range(len(words)-n+1)]
        all_ngrams.extend(ngrams)
    if len(all_ngrams) == 0:
        return 0
    return len(set(all_ngrams)) / len(all_ngrams)

In [ ]:
# Generate responses to varied prompts
test_prompts = [
    "What's your favourite food?",
    "Tell me something interesting.",
    "How do computers work?",
    "What makes a good friend?",
    "Why is the sky blue?",
    "What would you do with a million pounds?",
    "How do you deal with stress?",
    "What's the most beautiful place on Earth?",
]

all_responses = []
for p in test_prompts:
    r = chat_single_turn(p)
    all_responses.append(r)
    print(f"Q: {p}")
    print(f"A: {r}\n")

In [ ]:
# Compute diversity metrics
d1 = distinct_n(all_responses, 1)
d2 = distinct_n(all_responses, 2)

print(f"Distinct-1 (unigram diversity): {d1:.3f}")
print(f"Distinct-2 (bigram diversity):  {d2:.3f}")
print()
print("Interpretation:")
print("  > 0.8 Distinct-1 and > 0.6 Distinct-2 = healthy diversity")
print("  Low scores = repetitive, template-like responses")

**Distinct-N** is simple but effective. However, high diversity alone isn't sufficient — random text would score perfectly. That's why multiple metrics are always used together, and human evaluation remains essential.

## 8.3 Other Common Metrics

| Metric | What it measures | Limitation |
|--------|-----------------|------------|
| **Perplexity** | How "surprised" the model is by text — lower is better | Doesn't measure response *quality* |
| **BLEU** | N-gram overlap with reference responses | Penalises valid but differently-worded responses |
| **BERTScore** | Semantic similarity using BERT embeddings | Requires reference responses |
| **Human evaluation** | Direct human judgement | Expensive, slow, subjective |

No single metric captures "good conversation." This is an active area of research.

## 8.4 Exercise: Effect of Temperature on Quality

Run the cell below to generate responses at different temperatures, then manually evaluate: which temperature produces the best balance of quality and diversity?

In [ ]:
# Generate responses at different temperatures
prompt = "What advice would you give to a university student?"
print(f"Prompt: '{prompt}'\n")

for temp in [0.3, 0.5, 0.7, 0.9, 1.2]:
    responses = [chat_single_turn(prompt, temperature=temp) for _ in range(3)]
    d1 = distinct_n(responses, 1)
    print(f"Temperature {temp}:")
    for i, r in enumerate(responses):
        print(f"  {i+1}. {r}")
    print(f"  Distinct-1: {d1:.3f}")
    print()

**Discussion questions:**
- At which temperature are responses most coherent?
- At which temperature are responses most diverse?
- Where is the "sweet spot" — diverse enough to be interesting, coherent enough to be useful?

# 9. Limitations and Ethical Considerations

## 9.1 Known Limitations

Pre-trained chatbot models like DialoGPT have significant limitations:

**Hallucination:** The model generates *statistically plausible* text, not verified facts. It will confidently produce incorrect information. It doesn't "know" facts — it predicts token sequences that are likely given the training data.

**Context window:** Limited memory means long conversations lose early context. GPT-2/DialoGPT can only attend to ~1,024 tokens. Modern models (GPT-4: 128k, Claude: 200k) are much larger but still finite.

**Training data bias:** DialoGPT was trained on Reddit, which skews towards English-speaking, young, male, tech-oriented users. The model reflects these demographics in its responses — from cultural assumptions to potentially toxic language patterns.

**No genuine understanding:** The model doesn't "think" or "reason." It predicts the next token based on statistical patterns. It cannot perform reliable multi-step logic or maintain consistent beliefs.

## 9.2 Responsible Deployment

When deploying chatbots in production:

- **Content filtering** — Safety layers to detect and block harmful outputs.
- **Transparency** — Always disclose that users are talking to AI, not a human.
- **Scope limitation** — Restrict the chatbot to its intended domain.
- **Human escalation** — Clear path to a human agent.
- **Monitoring** — Continuous logging and review for problematic patterns.
- **Regular retraining** — Language evolves; the model must be kept current.

## 9.3 The Evolution Towards Safer Models

Modern systems address these limitations through:

| Technique | What it does |
|-----------|-------------|
| **RLHF** | Trains models to refuse harmful requests and acknowledge uncertainty |
| **Constitutional AI** | Defines principles the model should follow and trains self-correction |
| **RAG (Retrieval-Augmented Generation)** | Grounds responses in retrieved documents to reduce hallucination |
| **Tool use** | Lets models call calculators, databases, and search engines for factual accuracy |
| **Guardrails** | External systems that filter inputs and outputs for safety |

These advances explain why modern chatbots feel significantly more reliable than DialoGPT, despite sharing the same underlying Transformer architecture.

## 9.4 Discussion Exercise

Consider the following scenario:

> *A university deploys a DialoGPT-based chatbot on their admissions website to answer prospective students' questions about courses, fees, and campus life.*

In small groups, discuss:
1. What could go wrong?
2. What safeguards would you implement?
3. When should the chatbot hand over to a human?
4. How would you monitor the chatbot after deployment?

# 10. Summary

In this notebook we have:

1. **Explored the Transformer architecture** — self-attention, positional encoding, and the three model families (encoder, decoder, encoder-decoder).
2. **Deep-dived into tokenisation** — sub-word tokenisation, vocabulary sizes, the word-to-token ratio, and how different text styles tokenise differently.
3. **Visualised token probabilities** — understanding how language models assign probabilities, how entropy reflects confidence, and how context changes the distribution.
4. **Compared decoding strategies** — greedy, temperature, top-k, and nucleus sampling with visual explanations of how each shapes the probability distribution.
5. **Built a functional chatbot** — single-turn, multi-turn with conversation history, context truncation, and token counting.
6. **Compared GPT-2 vs DialoGPT** — demonstrating the impact of fine-tuning on conversational ability.
7. **Explored prompt engineering** — persona-based prompting and the evolution from early models to instruction-following systems.
8. **Evaluated chatbot quality** — qualitative criteria, Distinct-N diversity metrics, and the effect of temperature on output quality.
9. **Discussed ethics and safety** — hallucination, bias, deployment responsibility, and modern safety techniques.

## Key Takeaway

The Transformer architecture is the foundation of all modern language AI. Understanding how tokens are processed, how probabilities are computed, and how decoding strategies shape output gives you the conceptual toolkit to work with *any* language model — from the 124M-parameter GPT-2 we used today to the trillion-parameter systems powering commercial chatbots.

## Further Reading

- [Vaswani et al. (2017) — Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [Radford et al. (2019) — Language Models are Unsupervised Multitask Learners (GPT-2)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- [Zhang et al. (2020) — DialoGPT](https://arxiv.org/abs/1911.00536)
- [The Illustrated Transformer — Jay Alammar](https://jalammar.github.io/illustrated-transformer/)
- [Hugging Face Transformers Documentation](https://huggingface.co/docs/transformers/)
- [Ouyang et al. (2022) — InstructGPT / RLHF](https://arxiv.org/abs/2203.02155)
- [Holtzman et al. (2020) — The Curious Case of Neural Text Degeneration (Nucleus Sampling)](https://arxiv.org/abs/1904.09751)